# Stage 2 — scikit-learn で分類モデルを1本完走する

このノートは medguide-rag プロジェクトの **Stage 2**（機械学習の一連の流れを一通り体験する）の成果物です。
Stage 1 で前処理・可視化した **UCI Heart Disease（Cleveland）** の心臓病データ（欠損除外後 297件）を入力に、
**scikit-learn のロジスティック回帰**で「心疾患あり/なしを当てる分類モデル」を1本、
**学習 → 評価 → 簡易チューニング**まで通します。

> **このノートの読み方（Pythonが分からなくても妥当性を判断できる工夫）**
> 最後の「⑤ 係数と医学的定説の照合」に**照合スコアカード**（1枚の図）を置いています。
> モデルが学習した「どの項目がどちら向きに効くか（係数の符号）」を、医学の**定説**（出典付き・別途調査）と
> 突き合わせ、向きが一致していれば「モデルが常識と大きく矛盾していない」傍証になります。
> 逆向きなら、コードを読めなくても「データの取り違え・リーク・過学習」を疑えます。

> **ロジスティック回帰を選んだ理由**: 係数に**符号（向き）**があり、Stage 1 で確立した定説の向き
> （年齢↑で疾患↑・最大心拍数↑で疾患↓ 等）と直接照合できます。ランダムフォレスト等の「重要度」は
> 強さだけで向きが無いため、この健全性チェックには符号を持つ線形モデルが向いています。

> **注意**: 本ノートは学習・ポートフォリオ用途で、医療上の助言ではありません。
> データで得た係数は母集団の因果を証明しません（相関≠因果・単一データセットの限界）。

## このノートで通す「機械学習の流れ」

| 段階 | やること | 主に使う道具 |
|---|---|---|
| ① 準備 | 前処理して特徴量Xと正解yを作る | pandas |
| ② 分割 | 訓練用とテスト用に分ける・標準化 | train_test_split / StandardScaler |
| ③ 学習・評価 | モデルを学習し指標で測る | LogisticRegression / classification_report |
| ④ チューニング | 正則化の強さCを探して比較 | GridSearchCV / cross_val_score |
| ⑤ 照合 | 係数の向きを医学の定説と突き合わせる | （妥当性チェック・スコアカード） |


## ① データ準備 — 特徴量Xと正解yを作る

Stage 1 と同じ前処理を再現します（`?` を欠損にして数値化 → 欠損行を除外 → `num` を 0/1 に二値化）。
そのうえで、モデルに与える**特徴量 X** と、当てにいく**正解 y** に分けます。

> **⚠ 最重要（データリーク回避）**: 正解 `target` は `num`（0〜4の診断）から作ります。
> したがって `num` を特徴量に残すと「答えを見ながら答える」状態になり、精度が不自然に100%近くなります。
> **X からは `num` と `target` の両方を必ず除外**します。残る13列が特徴量です。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- Stage 1 と同じ前処理（欠損 '?' を除外し、num を 0/1 に二値化）---
DATA_PATH = Path("../data/sample/heart-disease-cleveland.csv")
df = pd.read_csv(DATA_PATH)
df_clean = df.replace("?", pd.NA).copy()
for col in ["ca", "thal"]:
    df_clean[col] = pd.to_numeric(df_clean[col])
df_clean = df_clean.dropna().reset_index(drop=True)
df_clean["target"] = (df_clean["num"] > 0).astype(int)  # 0=疾患なし, 1以上=疾患あり

print("前処理後の行数:", len(df_clean))
print("疾患なし(0)/あり(1):")
print(df_clean["target"].value_counts().sort_index().to_string())

### 列名の日本語対応表（図・表の表示用）

図や表の軸ラベルに使うため、英語列名と日本語の意味を対応づけます。データ本体は英語のまま扱い、表示時だけ日本語にします。


In [ ]:
JP = {
    "age": "年齢", "sex": "性別", "cp": "胸痛タイプ", "trestbps": "安静時血圧",
    "chol": "コレステロール", "fbs": "空腹時血糖", "restecg": "安静時心電図",
    "thalach": "最大心拍数", "exang": "運動誘発狭心症", "oldpeak": "ST低下",
    "slope": "ST傾き", "ca": "主要血管数", "thal": "サラセミア",
}
pd.DataFrame({"英語列名": list(JP.keys()), "日本語": list(JP.values())})

In [ ]:
# X と y に分ける（num と target はリーク源なので X から除外）
FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]
X = df_clean[FEATURES].copy()
y = df_clean["target"].copy()

print("特徴量 X の形 (行, 列):", X.shape)     # (297, 13)
print("正解 y の形         :", y.shape)
print("X に num が含まれない:", "num" not in X.columns)  # True であること

## ② 訓練/テスト分割と標準化 — 「未知データでの実力」を測る準備

モデルは学習に使ったデータでは良い点を出しがちです。本当の実力は**学習に使っていないデータ**で測ります。
そこで手元データを **訓練用（8割）とテスト用（2割）** に分けます。

- `stratify=y`: 分割後も「疾患あり/なしの比率」を訓練・テストで揃える（偏り防止）
- `random_state=42`: 乱数を固定し、何度実行しても同じ分割になる（再現性）

またロジスティック回帰は各特徴量の**スケール（単位）**に敏感です。年齢(数十)とST低下(0〜6)では桁が違うため、
**標準化**（平均0・分散1に揃える `StandardScaler`）で土俵を合わせます。標準化と学習を**1つの `Pipeline`** にまとめると、
「テストデータの情報が訓練の標準化に漏れる」事故（データリーク）を防げます。


In [ ]:
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# パイプラインをテキスト表示にする（Windowsの文字コード既定でHTML図生成が失敗するのを回避）
sklearn.set_config(display="text")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("訓練:", X_train.shape[0], "件 / テスト:", X_test.shape[0], "件")

# 標準化 → ロジスティック回帰 を1本のパイプラインに
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
print(pipe)

## ③ ベースラインモデルの学習と評価

まずは**チューニングなしの素のモデル**（ベースライン）を学習し、テストデータで測ります。
指標は次の4つ＋ROC-AUC を見ます。

| 指標 | 意味 | この課題での読み方 |
|---|---|---|
| 適合率 (precision) | 「疾患あり」と予測した中で本当に疾患ありの割合 | 誤って病気と言わない度合い |
| 再現率 (recall) | 本当に疾患ありの人を取りこぼさず当てた割合 | **見逃さない度合い（医療で最重要）** |
| F1 | 適合率と再現率の調和平均 | 両者のバランス |
| 正解率 (accuracy) | 全体で当たった割合 | 全体感（偏りに注意） |
| ROC-AUC | しきい値によらない判別力（0.5=当てずっぽう, 1.0=完璧） | モデルの総合的な分離能力 |

> **医療では再現率（recall）を重視**します。病気の人を「健康」と誤ると重大な見逃しになるためです。
> 正解率だけを見ると、偏ったデータで「多い方に全部寄せる」だけの無能なモデルを見逃すことがあります。


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

pipe.fit(X_train, y_train)                     # 学習
y_pred = pipe.predict(X_test)                  # 予測（0/1）
y_prob = pipe.predict_proba(X_test)[:, 1]      # 疾患ありの確率

print("=== ベースライン（テストデータでの成績）===")
print(classification_report(y_test, y_pred,
      target_names=["疾患なし", "疾患あり"], digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

### 図08: 混同行列 — 「どこで間違えたか」を4象限で見る

混同行列は「実際 × 予測」の件数表です。左上と右下（対角線）が正解、右上と左下が誤りです。
特に**左下（実際は疾患ありなのに『なし』と予測＝見逃し）**を小さくしたいのが医療の関心事です。


In [ ]:
import logging
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False
# Yu Gothic に bold ウェイトが無く出る「findfont: bold」警告を抑制（描画には影響なし）
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

cm = confusion_matrix(y_test, y_pred)
labels = ["疾患なし", "疾患あり"]

fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["予測:なし", "予測:あり"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["実際:なし", "実際:あり"])
thresh = cm.max() / 2
notes = [["正解(TN)", "誤検知(FP)"], ["見逃し(FN)", "正解(TP)"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}\n{notes[i][j]}", ha="center", va="center",
                color="white" if cm[i, j] > thresh else "#222222", fontsize=11)
ax.set_title("図08: 混同行列（テストデータ）")
fig.tight_layout()
fig.savefig(OUT / "08-confusion-matrix.png", dpi=100)
plt.show()

> **📊 図08の確認**: 対角線（左上＝TN・右下＝TP）が正解。左下 FN が「疾患ありの見逃し」で医療上もっとも避けたい誤り。
> この件数を見て、後段のしきい値調整や再現率重視の判断につなげます。

### 図09: ROC曲線 — しきい値によらない判別力（AUC）

「疾患あり」と判定する確率のしきい値を動かしながら、見逃しの少なさ（真陽性率）と誤検知（偽陽性率）の
トレードオフを描いた曲線です。左上に張り付くほど良く、曲線の下の面積 **AUC** が 1.0 に近いほど判別力が高いことを表します。


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(fpr, tpr, color="#0072B2", lw=2, label=f"ロジスティック回帰 (AUC={roc_auc:.3f})")
ax.plot([0, 1], [0, 1], color="#999999", lw=1, linestyle="--", label="当てずっぽう (AUC=0.5)")
ax.set_xlabel("偽陽性率（誤検知）")
ax.set_ylabel("真陽性率（見逃さない度合い）")
ax.set_title("図09: ROC曲線")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUT / "09-roc-curve.png", dpi=100)
plt.show()

> **📊 図09の確認**: 曲線が左上に寄り AUC が 0.5（対角線）より十分高ければ、モデルは疾患あり/なしを
> しきい値によらず判別できている。ベースライン時点での総合的な分離能力の確認。

## ④ 簡易チューニング — 正則化の強さ C を探す

ロジスティック回帰には**正則化**という「係数を大きくしすぎない（＝過学習を抑える）」仕組みがあり、
その強さをパラメータ **C** で調整します（C が小さいほど強く効かせて単純なモデル、大きいほど訓練データに忠実）。

どの C が良いかを、**交差検証**（訓練データをさらに5分割し、代わる代わる検証して平均する）で探します。
テストデータは最後の答え合わせ用に取っておき、C 選びには使いません（リーク回避）。
指標は ROC-AUC を最大化する C を選びます。


In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

param_grid = {"clf__C": [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)

print("最良の C   :", grid.best_params_["clf__C"])
print(f"交差検証AUC : {grid.best_score_:.3f}（訓練データ内5分割の平均）")

# ベースライン(C=1) と 最良モデル をテストデータで比較
best = grid.best_estimator_
base_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
best_auc = roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])
print("\n=== テストデータでの比較 ===")
print(f"ベースライン(C=1)  ROC-AUC: {base_auc:.3f}")
print(f"チューニング後      ROC-AUC: {best_auc:.3f}")

### 図10: 正則化の強さ C と交差検証スコア

横軸に C（対数目盛）、縦軸に交差検証の平均 ROC-AUC を取り、どの強さが良いかを可視化します。
山の頂上付近が「単純すぎず・複雑すぎず」の良い塩梅で、そこで選ばれた C が最良モデルになります。


In [ ]:
cv_means = grid.cv_results_["mean_test_score"]
cv_stds = grid.cv_results_["std_test_score"]
Cs = param_grid["clf__C"]
best_C = grid.best_params_["clf__C"]

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.errorbar(Cs, cv_means, yerr=cv_stds, marker="o", color="#0072B2", capsize=4)
ax.set_xscale("log")
ax.axvline(best_C, color="#009E73", linestyle="--", lw=1.5, label=f"最良 C = {best_C}")
ax.set_xlabel("正則化の強さ C（対数目盛・大きいほど訓練に忠実）")
ax.set_ylabel("交差検証 ROC-AUC（5分割平均）")
ax.set_title("図10: C とモデル性能の関係")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "10-tuning-C.png", dpi=100)
plt.show()

> **📊 図10の確認**: 縦棒は分割ごとのばらつき（標準偏差）。テストは60件と小さく単一の数値は揺れるため、
> 交差検証の平均で頑健性を確認している。C を変えても大きく崩れなければ、モデルは安定と読める。

## ⑤ 係数と医学的定説の照合（このノートの核）

ここが本ノートの**核**です。チューニング後のモデルが学習した**係数の符号**を、医学の**定説**（出典付き・別途調査）と
突き合わせます。標準化済みなので、係数が**正なら「その項目が大きいほど疾患ありに寄せる」**、負ならその逆を意味します。

**読み方**: 係数の向きが定説と一致していれば、前処理・特徴量の作り方・学習が大きく壊れていない傍証になります。
逆向き（下のスコアカードで⚠）なら、Pythonを読めなくても「データの取り違え・リーク・多重共線性」を疑えます。

> **補足（教材上の単純化）**: 胸痛タイプ・サラセミア・ST傾き・主要血管数などは本来カテゴリ変数ですが、
> ここでは数値（順序付き）として扱う単純化をしています。厳密には one-hot 符号化が望ましく、
> その場合カテゴリごとに係数が分かれます。符号照合はこの単純化のもとでの向き確認です。


In [ ]:
from matplotlib.patches import FancyBboxPatch, Rectangle

matplotlib.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

GREEN = "#009E73"    # Okabe-Ito: 一致
ORANGE = "#E69F00"   # Okabe-Ito: 逆向き（要確認）
INK = "#222222"; MUTE = "#555555"

# 学習した係数（標準化後・target=疾患あり を1とする向き）
clf = best.named_steps["clf"]
coef = pd.Series(clf.coef_[0], index=FEATURES)

# 照合対象と「定説が期待する係数の符号」（+1=大きいほど疾患 / -1=大きいほど疾患なし）
#  定説の向き・確度・出典は別途調査（research-analyst）。詳細出典は下の表に記載。
CONSENSUS = [
    ("age",     +1, "加齢で疾患リスク上昇 ↑",              "高"),
    ("sex",     +1, "男性が高リスク（1=男性）↑",           "高"),
    ("cp",      -1, "定説は典型的狭心症ほど疾患↑(値小)",     "中"),
    ("thalach", -1, "心拍が上がらないほど疾患 ↓",           "高"),
    ("exang",   +1, "運動誘発狭心症ありは疾患 ↑",           "高"),
    ("oldpeak", +1, "ST低下が大きいほど心筋虚血 ↑",         "高"),
    ("ca",      +1, "造影の狭窄血管数が多いほど疾患 ↑※",     "高"),
    ("thal",    +1, "欠損型(値大)は疾患 ↑",                "高"),
]

rows = []
n_match = 0
for feat, exp_sign, cons_txt, conf in CONSENSUS:
    c = coef[feat]
    actual = "↑（正）" if c > 0 else "↓（負）"
    match = (c > 0 and exp_sign > 0) or (c < 0 and exp_sign < 0)
    n_match += int(match)
    verdict = "✓ 一致" if match else "⚠ 逆向き"
    data_txt = f"係数 {c:+.2f}  {actual}"
    rows.append((JP[feat], data_txt, cons_txt, f"確度:{conf}", verdict, match))

fig, ax = plt.subplots(figsize=(11.5, 5.6))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

ax.text(0.012, 0.965, "係数と医学的定説との照合スコアカード", fontsize=15, fontweight="bold", color=INK, va="center")
summary_color = GREEN if n_match == len(rows) else ORANGE
ax.text(0.012, 0.915, f"照合 {len(rows)} 項目中 {n_match} 項目で係数の向きと医学の定説が一致（{n_match} / {len(rows)}）",
        fontsize=11.5, color=summary_color, fontweight="bold", va="center")

X = {"item": 0.015, "data": 0.185, "consensus": 0.44, "conf": 0.775, "judge": 0.86}
hy = 0.835
ax.text(X["item"], hy, "項目", fontsize=10.5, fontweight="bold", color=MUTE, va="center")
ax.text(X["data"], hy, "モデルの係数（向き）", fontsize=10.5, fontweight="bold", color=MUTE, va="center")
ax.text(X["consensus"], hy, "医学の定説", fontsize=10.5, fontweight="bold", color=MUTE, va="center")
ax.text(X["conf"], hy, "確度", fontsize=10.5, fontweight="bold", color=MUTE, va="center")
ax.text(X["judge"] + 0.062, hy, "判定", fontsize=10.5, fontweight="bold", color=MUTE, va="center", ha="center")
ax.plot([0.012, 0.985], [0.80, 0.80], color="#cccccc", lw=1)

row_top = 0.74; row_h = 0.088
for i, (item, data_txt, cons_txt, conf_txt, verdict, match) in enumerate(rows):
    yc = row_top - i * row_h
    if i % 2 == 1:
        ax.add_patch(Rectangle((0.012, yc - row_h / 2), 0.973, row_h,
                               facecolor="#f5f5f5", edgecolor="none", zorder=0))
    ax.text(X["item"], yc, item, fontsize=11, fontweight="bold", color=INK, va="center")
    ax.text(X["data"], yc, data_txt, fontsize=10, color=INK, va="center")
    ax.text(X["consensus"], yc, cons_txt, fontsize=10, color=INK, va="center")
    ax.text(X["conf"], yc, conf_txt, fontsize=9.5, color=MUTE, va="center")
    badge = GREEN if match else ORANGE
    ax.add_patch(FancyBboxPatch((X["judge"], yc - 0.032), 0.122, 0.064,
                                boxstyle="round,pad=0.004,rounding_size=0.02",
                                facecolor=badge, edgecolor="none", zorder=2))
    ax.text(X["judge"] + 0.061, yc, verdict, fontsize=10, fontweight="bold",
            color="white", ha="center", va="center", zorder=3)

ax.text(0.012, 0.02,
        "※ 判定は「色＋記号(✓/⚠)＋文字」を併記（色覚・グレースケールでも判別可）。向きの一致は健全性の傍証で、相関≠因果（証明ではない）。",
        fontsize=8.5, color=MUTE, va="center")

fig.savefig(OUT / "11-coef-scorecard.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"照合: {n_match}/{len(rows)} 項目が定説と向き一致")

> このスコアカードが結論の要約です。判定バッジは **色（緑=一致／橙=逆向き）＋記号（✓/⚠）＋文字** を併記し、
> 色が見えにくい環境でも読めるようにしています。逆向き（⚠）が出た項目は、下の但し書き（多重共線性）を参照してください。

### スコアカードの読み解き — ⚠（逆向き）が出た項目こそ学びどころ

健全性チェックは「⚠ が出た＝失敗」ではありません。**⚠ は『定説と違う向きなので理由を調べよ』という合図**です。
特に **胸痛タイプ（cp）** は、教科書の定説とこのデータセットの実際の向きが**既知の逆転**を起こす代表例です。

- **教科書の定説**: 典型的狭心症（症状が典型的）ほど冠動脈疾患ありの方向で、無症候はもっとも疾患なしの方向（Diamond-Forrester 型の事前確率モデル）。データ上 `cp` は値が小さいほど典型的狭心症。→ 定説の期待は「係数は負」。
- **このデータの実際**: Cleveland データでは無症候（cp=4）が疾患ありと強く相関するため、モデルの係数は**正**に出やすく、定説と**逆向き（⚠）**になります。
- **なぜ逆転するか（推測・断定しない）**: 本データは「胸痛を主訴に来院し血管造影を受けた患者」の後ろ向きコホートで、無症候カテゴリは糖尿病・高齢者の**無症候性心筋虚血**が造影に紹介された可能性があります（紹介バイアス）。**これは仮説であり、このデータで検証された因果ではありません。**

→ つまり ⚠ は「バグ」ではなく「このデータ特有のクセを健全性チェックが炙り出した」好例です。コードを読めなくても、
結論（係数の向き）を定説と突き合わせるだけで“調べるべき点”に気づける、という本ノートの狙いそのものです。

### 定説の出典（research-analyst 調査・出典付き）

1. **年齢** — 加齢は冠動脈疾患の主要な非修正性危険因子。確度: 高
   - [StatPearls: Risk Factors for Coronary Artery Disease (NCBI)](https://www.ncbi.nlm.nih.gov/books/NBK554410/)
2. **性別** — 中高年期は男性のリスクが高い。確度: 高
   - [Sex Differences in Coronary Heart Disease (Circulation)](https://www.ahajournals.org/doi/10.1161/01.CIR.95.1.252)
3. **胸痛タイプ (cp)** — 定説は「典型的狭心症ほど疾患あり／無症候ほど疾患なし」だが、**本データは逆転**（上記参照）。確度: 中（症状分類の定説自体は中〜高／データとの整合は既知の矛盾あり）
   - [2021 AHA/ACC Guideline for the Evaluation and Diagnosis of Chest Pain (JACC)](https://www.jacc.org/doi/10.1016/j.jacc.2021.07.052)
   - [Detrano R, et al. Am J Cardiol 1989（原著データ由来）](https://www.ajconline.org/article/0002-9149(89)90524-9/fulltext)
4. **最大心拍数 (thalach)** — 変時性不全（運動時に心拍が上がらない）は疾患・予後の予測因子。確度: 高
   - [Chronotropic Incompetence (PMC)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3065291/)
5. **運動誘発狭心症 (exang)** — exang=1 は冠動脈疾患ありの方向。確度: 高
   - [Treadmill Stress Testing (StatPearls)](https://www.ncbi.nlm.nih.gov/books/NBK499903/)
   - [Exercise Stress Testing (AAFP)](https://www.aafp.org/pubs/afp/issues/2017/0901/p293.html)
6. **ST低下 (oldpeak)** — 運動誘発ST低下が大きいほど心筋虚血・重症の方向。確度: 高
   - [ST Segment Depression (ScienceDirect Topics)](https://www.sciencedirect.com/topics/biochemistry-genetics-and-molecular-biology/st-segment-depression)
7. **主要血管数 (ca) ※** — 造影で描出される狭窄血管が多いほど多枝病変＝疾患ありかつ重症。確度: 高
   - [Multivessel disease（定義・予後・PubMed）](https://pubmed.ncbi.nlm.nih.gov/1901190/)
   - **※ 注意**: `ca` は目的変数（血管造影で50%以上狭窄=疾患あり）と**同一の検査（冠動脈造影）由来**のため、相関というより定義上ほぼ直結（tautological）。因果の例には使えず、あくまで検査所見同士の整合性チェックとして扱う。
8. **サラセミア/心筋血流 (thal)** — reversible/fixed defect はいずれも normal より疾患ありの方向。確度: 高
   - [Stress-Induced Thallium Defects (Circulation)](https://www.ahajournals.org/doi/10.1161/01.cir.98.6.501)
   - **注意**: UCI 符号化は 3=正常, 6=固定欠損, 7=可逆性欠損。コード値の大小に医学的序列はなく、本来カテゴリ変数（数値扱いは教材上の単純化）。

> **重要な但し書き（相関≠因果・多重共線性）**: 上記の定説は母集団レベルの一般的知見で、今回の297件で学習した係数が
> 定説を「証明」するわけではありません。線形モデルの係数は特徴量どうしが相関している（多重共線性）と符号が入れ替わることがあり、
> スコアカードで⚠（逆向き）が出てもただちに「バグ」とは限らず、共線性・データ固有のクセ・カテゴリの数値扱いを疑うのが正しい読み方です。
> 本節はあくまで「モデルが常識と大きく矛盾していないか」の**健全性チェック**であり、医療上の結論ではありません。


## まとめ — 分かったことと次への橋渡し

機械学習の基本的な流れ（準備 → 分割・標準化 → 学習・評価 → チューニング → 定説照合）を、
ロジスティック回帰で1本通しました。

### やったこと
1. **準備**（①）: 前処理済データから特徴量13列と正解を作成。`num` を除外しデータリークを回避
2. **分割・標準化**（②）: 層化分割 + `Pipeline` で標準化と学習を一体化（リーク防止）
3. **学習・評価**（③）: ベースラインを学習し、適合率・再現率・F1・正解率・ROC-AUC＋混同行列・ROC曲線で評価
4. **チューニング**（④）: 交差検証で正則化 C を探索し、ベースラインと比較
5. **照合**（⑤）: 係数の符号を医学の定説と突き合わせ、健全性を確認（スコアカード）

### 次のステージへ
Stage 3 では、いよいよ本題の **医療ガイドライン RAG**（文書検索 + LLM）に進みます。
本 Stage で「教師あり学習の一連の流れ」と「結果を定説で検算する型」を確立したことが、
RAG の回答品質評価（Stage 4）でも活きます。
